
## OpenTheChestsGym: Gym-Compatible Symbolic RL Environment

The `OpenTheChestsGym` class provides a **Gymnasium-compatible wrapper** around the symbolic event-based environment `OpenTheChests`. This integration enables the use of **standard reinforcement learning libraries** such as Stable Baselines3, RLlib, etc., while preserving the symbolic and temporal reasoning capabilities of the underlying environment.

---

### What It Does

* Wraps `OpenTheChests` into a standard `gym.Env` interface.
* Handles **observation formatting**, **action space definition**, and **step/reset** lifecycle.
* Supports:

  * Discrete or multi-binary **action spaces** (single int vs. one-hot vector).
  * Flattened SB3-friendly **observation dictionaries**.

---

### Observation Format

The observation is returned as a flat dictionary with the following keys:

| Key        | Description                                     | Type          |
| ---------- | ----------------------------------------------- | ------------- |
| `active`   | Binary vector indicating which boxes are active | `MultiBinary` |
| `open`     | Binary vector indicating which boxes are open   | `MultiBinary` |
| `e_type`   | Encoded type of the observed event              | `Discrete`    |
| Attributes | One `Discrete` field per symbolic attribute     | `Discrete`    |
| `start`    | Start time of the observed event                | `Box(1,)`     |
| `end`      | End time of the observed event                  | `Box(1,)`     |
| `duration` | Duration of the event (end - start)             | `Box(1,)`     |

If `stb3=True`, this format is **flattened** and compatible with Stable Baselines 3.

---

### Action Format

You can control how the agent presses buttons:

* `discrete=True`:
  Agent sends a **single integer** encoding all button presses (as a binary number).
  Example: `3` = `0b11` → press both buttons.

* `discrete=False`:
  Agent sends a **list of 0s and 1s** of length equal to number of boxes.
  Example: `[0, 1, 1]` = press box 1 and 2.

---

### Usage Tips

* You can create environments manually or using `from_config_file(...)` with a YAML configuration.
* Use `env.get_otc()` to access the underlying `OpenTheChests` class for advanced behavior or visualization.
* You can inspect all types and attributes using:

  ```python
  env.get_types()
  env.get_attributes()
  ```

---

### Config-Based Initialization

To initialize the Gym environment from a YAML config:

```python
from openthechests.openthechests.src.OpenTheChestsGym import OpenTheChestsGym

env = OpenTheChestsGym.from_config_file(
    env_config_file="path/to/config.yaml",
    pattern_configs_folder="path/to/patterns",
    stb3=True,
    discrete=True,
    verbose=False
)
```

In [1]:
from openthechests.src.OpenTheChestsGym import OpenTheChestsGym
import numpy as np

# Define a simple instruction set with 2 boxes and a basic temporal pattern
instructions = [
    [
        {"command": "delay", "parameters": 10},
        {"command": "noise", "parameters": 0},
        {"command": "instantiate", "parameters": ("A", {"bg": "red", "fg": "white"}, {"mu": 2, "sigma": 0.1}), "variable_name": "e1"},
        {"command": "instantiate", "parameters": ("A", {"bg": "blue", "fg": "white"}, {"mu": 5, "sigma": 0.1}), "variable_name": "e2"},
        {"command": "after", "parameters": ["e2", "e1"], "variable_name": "e2", "other": {"gap_dist": {"mu": 3, "sigma": 0.1}}}
    ],
    [
        {"command": "delay", "parameters": 12},
        {"command": "noise", "parameters": 0},
        {"command": "instantiate", "parameters": ("B", {"bg": "yellow", "fg": "black"}, {"mu": 1, "sigma": 0.1}), "variable_name": "e1"},
        {"command": "instantiate", "parameters": ("B", {"bg": "green", "fg": "black"}, {"mu": 4, "sigma": 0.1}), "variable_name": "e2"},
        {"command": "after", "parameters": ["e2", "e1"], "variable_name": "e2", "other": {"gap_dist": {"mu": 2, "sigma": 0.1}}}
    ]
]

# Define symbolic types and attributes
all_event_types = ["A", "B"]
all_event_attributes = {"bg": ["red", "blue", "yellow", "green"], "fg": ["white", "black"]}
all_noise_types = []
all_noise_attributes = {}

# Create environment (non-discrete = MultiBinary actions, stb3 = True for compatibility)
env = OpenTheChestsGym(
    instructions=instructions,
    all_event_types=all_event_types,
    all_event_attributes=all_event_attributes,
    all_noise_types=all_noise_types,
    all_noise_attributes=all_noise_attributes,
    discrete=False,  # Action is [0, 1] style
    stb3=True,
    verbose=True
)

# Reset environment
obs, _ = env.reset()
done = False
step = 0

print("\n--- Starting Random Action Loop ---\n")

while not done:
    step += 1
    print(f"\nStep {step}")

    # Sample random binary action (e.g., [0, 1] for pressing box 1)
    action = env.action_space.sample()
    print(f"Random action: {action}")

    # Step through the environment
    obs, reward, done, _, _ = env.step(action)

    # Print outcome
    print(f"Reward: {reward}")
    print(f"Done: {done}")
    print(f"Observation: {obs}")

print("\n Episode finished")


All event types : ['A', 'B']
All noise types : []
All event attributes : {'bg': ['red', 'blue', 'yellow', 'green'], 'fg': ['white', 'black']}
All noise attributes : {}
Initialising 2 boxes.
Starting Reset
Sampling new events from pattern 0: [Event(type='A', attr={'bg': 'red', 'fg': 'white'}, start=1.515, end=3.599), Event(type='A', attr={'bg': 'blue', 'fg': 'white'}, start=6.699, end=11.799)]
Sampling new events from pattern 1: [Event(type='B', attr={'bg': 'yellow', 'fg': 'black'}, start=6.566, end=7.466), Event(type='B', attr={'bg': 'green', 'fg': 'black'}, start=9.439, end=13.536)]
Making one internal step to get context and advance timeline.
Active timeline [Event(type='A', attr={'bg': 'red', 'fg': 'white'}, start=1.515, end=3.599), Event(type='B', attr={'bg': 'yellow', 'fg': 'black'}, start=6.566, end=7.466)]
The last observed event ends at 3.599
Advancing _time to 3.599
Observing context Event(type='A', attr={'bg': 'red', 'fg': 'white'}, start=1.515, end=3.599)
Activating box 0.
R

In [2]:
import gym

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
